### Exercise 5: Combine and implement the reusable code from previous exercises

In [ ]:
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Any, Dict, List, Optional, Union
import pandas as pd

#### Boilerplate Code Template Implementation for entire Pipeline

In [ ]:
class DataPipeline(ABC):

    def __init__(self, source: Path, destination: Path) -> None:
        self.source = source
        self.destination = destination
        self.df: pd.DataFrame | None = None

    @abstractmethod
    def load(self) -> None: ...

    @abstractmethod
    def clean(self) -> None: ...

    @abstractmethod
    def transform(self) -> None: ...

    @abstractmethod
    def save(self) -> None: ...

    @log_action
    @log_time
    def run(self) -> None:
        LOGGER.info("PIPELINE START: %s to %s", self.source, self.destination)
        self.load()
        self.clean()
        self.transform()
        self.save()
        LOGGER.info("PIPELINE END: result saved in %s", self.destination)

#### Subclass Core Logics Rule Abstract from DataPipeline

In [ ]:
class CSVDataPipeline(DataPipeline):

    def __init__(self, source: Path, destination: Path,
                 cleaner: DataCleaner, transforms: list[DataTransform]) -> None:
        super().__init__(source, destination)
        self.reader = CSVReader(source)
        self.cleaner = cleaner
        self.transforms = list(transforms)

    def load(self) -> None:
        self.df = self.reader.read()
        self.reader.preview(3)

    def clean(self) -> None:
        self.df = self.cleaner.clean(self.df)

    def transform(self) -> None:
        for transform in self.transforms:
            LOGGER.info("Transform: %s", type(transform).__name__)
            self.df = transform.apply(self.df)

    def save(self) -> None:
        self.df.to_csv(self.destination, index=False)

In [ ]:
def build_pipeline(strategy: MissingValueStrategy, transform_names: list[str]) -> CSVDataPipeline:
    return CSVDataPipeline(
        source=INPUT_PATH,
        destination=OUTPUT_PATH,
        cleaner=DataCleaner(strategy),
        transforms=[TransformFactory.create(name) for name in transform_names],
    )

In [ ]:
pipeline = build_pipeline(
    strategy=FillMean(),
    transform_names=["remove_duplicates", "standardize_text", "normalize"],
)

LOGGER.info("[INFO]: Collaborators of the pipeline:")
LOGGER.info("  reader     : %s", type(pipeline.reader).__name__)
LOGGER.info("  cleaner    : %s to %s", type(pipeline.cleaner).__name__,
            type(pipeline.cleaner.strategy).__name__)
LOGGER.info("  transforms : %s",
            [type(t).__name__ for t in pipeline.transforms])

pipeline.run()